Excercise 1

In [80]:
import numpy as np

In [81]:
#Probabilities and components-----------------------------------------

P_x = [0.985, 0.015] #not faulty, faulty
P_x_z = [1/3, 1] #not faulty, faulty
k = 10 #number of steps

In [82]:
#Define bayes filter function-----------------------------------------
def bayes_filter(x_z, x):
    #bayes filter without normalization
    P_not_faulty_num = x_z[0]*x[0] #faulty
    P_faulty_num = x_z[1]*x[1]     #not faulty

    #Normalization
    norm = P_not_faulty_num + P_faulty_num

    #bayes filter with normalization
    P_not_faulty = P_not_faulty_num/norm
    P_faulty = P_faulty_num/norm
    
    return P_not_faulty, P_faulty

In [129]:
#Test bayes filter function-----------------------------------------
values = []

prior = bayes_filter(P_x_z, P_x) 
values.append(f"Filtered Estimate for k=1: [{prior[0]} {prior[1]}]")

for i in range(2,k+1):
    new_prior = bayes_filter(P_x_z, prior)
    values.append(f"Filtered Estimate for k={i}: [{new_prior[0]} {new_prior[1]}]")
    prior = new_prior

print('\n'.join(values)) #make new line for each value

Filtered Estimate for k=1: [0.9563106796116504 0.043689320388349516]
Filtered Estimate for k=2: [0.8794642857142857 0.12053571428571432]
Filtered Estimate for k=3: [0.7086330935251798 0.2913669064748202]
Filtered Estimate for k=4: [0.44772727272727264 0.5522727272727274]
Filtered Estimate for k=5: [0.21274298056155502 0.7872570194384451]
Filtered Estimate for k=6: [0.08263422818791942 0.9173657718120806]
Filtered Estimate for k=7: [0.029150636282923924 0.9708493637170761]
Filtered Estimate for k=8: [0.00990945674044265 0.9900905432595574]
Filtered Estimate for k=9: [0.0033251189953752134 0.9966748810046248]
Filtered Estimate for k=10: [0.0011108354384698658 0.9988891645615301]


Excercise 2 Part A

In [84]:
#Probabilities and components-----------------------------------------

#Current robot room
P_x0 = [1, 0, 0] #room 1, room 2, room 3
transition = np.matrix([[0, 0, 0], [0.5, 0.8, 0.3], [0.5, 0.2, 0.7]]).T #(room 1, room 2, room 3) x (room 1, room 2, room 3)

steps = 10 #number of steps

In [130]:
#Define bayes prediction function-----------------------------------------
def bayes_prediction(init_prior, t_matrix):
    #bayes filter without normalization
    init_posterior = init_prior @ t_matrix # 
    
    return init_posterior


In [111]:
prediction = []

prediction_prior = bayes_prediction(P_x0, transition)
prediction.append(f"Prob Distrubution for k=1: [{prediction_prior}]")

for i in range(2,steps+1):
    new_prediction = bayes_prediction(prediction_prior, transition)
    prediction.append(f"Prob Distrubution for k={i}: [{new_prediction}]")
    
    prediction_prior = new_prediction

print('\n'.join(prediction)) #make new line for each value

Prob Distrubution for k=1: [[[0.  0.5 0.5]]]
Prob Distrubution for k=2: [[[0.   0.55 0.45]]]
Prob Distrubution for k=3: [[[0.    0.575 0.425]]]
Prob Distrubution for k=4: [[[0.     0.5875 0.4125]]]
Prob Distrubution for k=5: [[[0.      0.59375 0.40625]]]


Excercise 2 Part B

In [ ]:
M = np.asarray([[0, 0.5, 0.5],[0, 0.9, 0.1],[0, 0.1, 0.9]])  # measurement model
T = np.asarray([[0.0, 0.0, 0.0],[0.5, 0.8, 0.3],[0.5, 0.2, 0.7]])  # transition model
initial_state = np.asarray([1, 0, 0])  # initial state of the world
observations = np.asarray([1, 2, 3, 3, 2, 3])  # history of observations
observation_idx = observations - 1  # convenience for indexing
steps = 5

alpha = initial_state * M[:,observations[1]]  # initial alpha value
beta = np.asarray([1.0, 1.0, 1.0])  # initial beta value

[0.5 0.1 0.9]


In [ ]:
def forward_step(alpha_k, Mmat, Tmat, step):
    """Computes the forward step for a Bayes Filter.
    
    Inputs:
        - alpha_k: previous alpha vector for state k
        - Mmat: measurement matrix
        - Tmat: transition matrix
        - step: the state value k that is being assessed
    
    Outputs:
        - alpha_kplus1: updated alpha vector for state k+1
    """
    for k in range(alpha_k):
        temp_sum = np.sum()
        alpha_kplus1 = Mmat[:,observations[step]] * temp_sum  # multiply by likelihood of the observation
    return(alpha_kplus1)

def backwards_step(beta_kplus1, Mmat, Tmat, step):
    """Computes the backward step for a Bayes Smoother.
    
    Inputs:
        - beta_kplus1: beta vector for future state k+1
        - Mmat: measurement matrix
        - Tmat: transition matrix
        - step: the state value k that is being assessed
    
    Outputs:
        - beta_k: updated beta vector for state k
    """
    beta_k = np.zeros_like(beta_kplus1)  # initialize a vector to compute beta_k
    for s in range(len(beta_kplus1)):  # for each possible state of x_k
        for q in range(len(beta_kplus1)):  # for each possible state of x_k+1
            beta_k[s] += beta_kplus1[q] * Tmat[s,q] * Mmat[q,observation_idx[-step-1]]  # sum over all possible transition-observation pairs
    return(beta_k)

In [127]:
print(f"Forward Step for k=1: {alpha}")

forward = []  # store each forward value for later smoothing
for i in range(steps):  # walk through state history
    f_alpha = forward_step(alpha, M, T, i)
    forward.append(f_alpha)

    print(f"Forward Step for k={i+2}: {f_alpha}")
    alpha = f_alpha


Forward Step for k=1: [0.5 0.  0. ]
Forward Step for k=2: [0.025 0.18  0.025]
Forward Step for k=3: [0.03725 0.0025  0.11745]
Forward Step for k=4: [0.0023625 0.008537  0.0603945]
Forward Step for k=5: [0.00182553 0.03346353 0.00304613]
Forward Step for k=6: [0.00678398 0.00025579 0.0199884 ]


In [62]:
inital_est = [forward[0][0]/np.sum(forward[0]), forward[0][1]/np.sum(forward[0]), forward[0][2]/np.sum(forward[0])]
print(f"Filtered Estimate for k=1: ", inital_est)

for i in range(steps):  # walk through state history
    normalized_alpha = forward[i+1]/np.sum(forward[i+1])
    print(f"Filtered Estimate for k={i+2}: ", normalized_alpha)

Filtered Estimate for k=1:  [np.float64(nan), np.float64(nan), np.float64(nan)]
Filtered Estimate for k=2:  [nan nan nan]
Filtered Estimate for k=3:  [nan nan nan]
Filtered Estimate for k=4:  [nan nan nan]
Filtered Estimate for k=5:  [nan nan nan]


C:\Users\mmiller\AppData\Local\Temp\ipykernel_138124\724711465.py:1: RuntimeWarning: invalid value encountered in scalar divide
  inital_est = [forward[0][0]/np.sum(forward[0]), forward[0][1]/np.sum(forward[0]), forward[0][2]/np.sum(forward[0])]
C:\Users\mmiller\AppData\Local\Temp\ipykernel_138124\724711465.py:5: RuntimeWarning: invalid value encountered in divide
  normalized_alpha = forward[i+1]/np.sum(forward[i+1])


In [49]:
backward = [beta]  # store each backward value for later smoothing

print(f"Backwards Step for k=5: ", beta)

for i in range(steps):  # walk through state history
    beta = backwards_step(beta, M, T, i)
    backward.append(beta)
    print(f"Backwards Step for k={steps-i}: ", beta)

Backwards Step for k=5:  [3.68544912e-06 5.03164462e-06 2.93722311e-06]
Backwards Step for k=4:  [1.70728864e-06 2.32319031e-06 1.35929900e-06]
Backwards Step for k=3:  [9.89677892e-07 4.23015668e-07 1.30889472e-06]
Backwards Step for k=2:  [6.55407148e-07 9.04738730e-07 4.96583041e-07]
Backwards Step for k=1:  [2.92422275e-07 3.99236272e-07 2.33054219e-07]


In [52]:
backward.reverse()  # re-orient the vector to be in the same order of states as the forward pass
for i, (a, b) in enumerate(zip(forward, backward)):  # get the corresponding alpha and beta for each state
    numerator = a * b
    smoothed = numerator / np.sum(numerator)
    print(f"Smoothed Value for k={i+1}: ", smoothed)

Smoothed Value for k=1:  [0.29417418 0.0151303  0.69069553]
Smoothed Value for k=2:  [0.03793442 0.19728712 0.76477846]
Smoothed Value for k=3:  [0.04602587 0.04006807 0.91390605]
Smoothed Value for k=4:  [0.03631285 0.91483722 0.04884993]
Smoothed Value for k=5:  [0.29417418 0.0151303  0.69069553]


Problem 2

In [ ]:
MM = np.asarray([[0.6, 0.4, 0],[0.3, 0.7, 0],[0, 0, 1]])  # measurement model
TM = np.asarray([[0.8, 0.2, 0],[0.4, 0.4, 0.2],[0.2, 0.6, 0.2]])  # transition model
initial_state = np.asarray([1, 0, 0])  # initial state of the world
observations = np.asarray([1])  # history of observations
observation_idx = observations - 1  # convenience for indexing
steps = 4